In [ ]:
# ── MASTER SETUP CELL — run once after every runtime restart ──

from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

# ── Install ───────────────────────────────────────────────────
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "pyannote.audio==3.1.1", "pyannote.metrics==3.2.1",
    "omegaconf==2.3.0", "soundfile", "scikit-learn"
], check=True)

# ── Patches (run ONCE on a clean import) ─────────────────────
import sys, importlib
from types import ModuleType
from unittest.mock import MagicMock
import numpy as np
import torchaudio

# numpy 2.0
np.NaN = np.nan
np.NAN = np.nan



import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

# torchaudio missing attributes
torchaudio.set_audio_backend = lambda x: None
torchaudio.get_audio_backend = lambda: "soundfile"

class _AudioMetaData:
    def __init__(self, sample_rate=0, num_frames=0,
                 num_channels=0, bits_per_sample=0, encoding=""):
        self.sample_rate     = sample_rate
        self.num_frames      = num_frames
        self.num_channels    = num_channels
        self.bits_per_sample = bits_per_sample
        self.encoding        = encoding

_backend_common = ModuleType("torchaudio.backend.common")
_backend_common.AudioMetaData = _AudioMetaData
_backend = ModuleType("torchaudio.backend")
_backend.common = _backend_common
sys.modules["torchaudio.backend"]        = _backend
sys.modules["torchaudio.backend.common"] = _backend_common
torchaudio.backend                       = _backend
torchaudio.AudioMetaData                 = _AudioMetaData

# speechbrain.pretrained renamed in v1.0
sys.modules["speechbrain.pretrained"] = MagicMock()

# huggingface_hub — safe patch with double-patch prevention
import huggingface_hub

if not getattr(huggingface_hub, '_patched_by_courtroom', False):
    _REAL_DOWNLOAD = huggingface_hub.hf_hub_download

    def _patched_download(*args, **kwargs):
        kwargs.pop("use_auth_token", None)
        return _REAL_DOWNLOAD(*args, **kwargs)

    huggingface_hub.hf_hub_download           = _patched_download
    huggingface_hub._patched_by_courtroom     = True
    print("hf_hub_download patched")
else:
    _patched_download = huggingface_hub.hf_hub_download
    print("hf_hub_download already patched — skipping")

if not hasattr(torchaudio, 'info'):
    def _torchaudio_info(path):
        import soundfile as sf
        info = sf.info(path)
        class _Info:
            sample_rate  = info.samplerate
            num_frames   = info.frames
            num_channels = info.channels
        return _Info()
    torchaudio.info = _torchaudio_info

# ── Import pyannote ───────────────────────────────────────
import pyannote.audio.core.pipeline as _pp
_pp.hf_hub_download = _patched_download

import pyannote.audio
print("pyannote:", pyannote.audio.__version__)

# ── Load pipeline ─────────────────────────────────────────────
import torch
from pyannote.audio import Pipeline

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

diar_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=os.environ["HF_TOKEN"]
).to(DEVICE)

print("Pipeline loaded on", DEVICE)

Mounted at /content/drive
hf_hub_download patched
pyannote: 3.1.1
Device: cuda


config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/399 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/221 [00:00<?, ?B/s]

Pipeline loaded on cuda


In [ ]:
from pyannote.core import Segment, Annotation
from pyannote.metrics.diarization import DiarizationErrorRate
import soundfile as sf
import csv, json
import numpy as np
from pathlib import Path

AUDIO_DIR   = Path("/content/drive/MyDrive/courtroom-diarizer-eval/audio")
RTTM_DIR    = Path("/content/drive/MyDrive/courtroom-diarizer-eval/annotations/dev")
OUT_DIR     = Path("/content/drive/MyDrive/courtroom-diarizer-eval/results")
OUT_DIR.mkdir(exist_ok=True)

# ── Batch config — change BATCH for each run ──────────────────
BATCH      = 1   # 1,2,3,4
BATCH_SIZE = 54
START      = (BATCH - 1) * BATCH_SIZE
END        = START + BATCH_SIZE
RESULTS_FILE = "voxconverse_batch1.csv"

# ── Load already-completed results ────────
results_path = OUT_DIR / RESULTS_FILE
existing     = []
completed    = set()
print(f"Starting batch {BATCH} fresh")

# ── Run this batch ────────────────────────────────────────────
metric    = DiarizationErrorRate(collar=0.25, skip_overlap=False)
wav_files = sorted(AUDIO_DIR.glob("*.wav"))[START:END]
results   = []

print(f"Batch {BATCH}/4 — files {START+1}–{min(END, 216)} ({len(wav_files)} files)")

for i, wav_path in enumerate(wav_files, START+1):
    if wav_path.stem in completed:
        print(f"[{i}/216] {wav_path.stem} — already done, skipping")
        continue

    rttm_path = RTTM_DIR / f"{wav_path.stem}.rttm"
    if not rttm_path.exists():
        continue

    try:
        diarization = diar_pipeline(str(wav_path))

        hyp = Annotation(uri=wav_path.stem)
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            hyp[turn, speaker] = speaker

        ref = Annotation(uri=wav_path.stem)
        with open(rttm_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 9 or parts[0] != "SPEAKER":
                    continue
                seg = Segment(float(parts[3]),
                              float(parts[3]) + float(parts[4]))
                ref[seg, parts[7]] = parts[7]

        components = metric(ref, hyp, detailed=True)
        total      = components.get("total", 0.0)
        der        = abs(metric)

        def rate(k):
            return components.get(k, 0.0) / total if total > 0 else 0.0

        info = sf.info(str(wav_path))
        results.append({
            "file":        wav_path.stem,
            "duration_s":  round(info.duration, 2),
            "DER":         round(der * 100, 2),
            "miss":        round(rate("missed detection") * 100, 2),
            "false_alarm": round(rate("false alarm") * 100, 2),
            "confusion":   round(rate("confusion") * 100, 2),
        })
        print(f"[{i}/216] {wav_path.stem} | DER={der*100:.1f}%")

        # Save after every file
        all_results = existing + results
        fields = ["file","duration_s","DER","miss","false_alarm","confusion"]
        with open(results_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fields)
            w.writeheader()
            w.writerows(all_results)

    except Exception as e:
        print(f"[{i}] FAILED: {wav_path.stem} — {e}")

print(f"\nBatch {BATCH} done. {len(results)} new files processed.")
print(f"Total saved to CSV: {len(existing) + len(results)}")

Starting batch 1 fresh
Batch 1/4 — files 1–54 (54 files)


/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)
/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


[1/216] abjxc | DER=0.2%


/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


[2/216] afjiv | DER=3.9%


...

/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


[52/216] ggvel | DER=5.4%


/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


[53/216] gocbm | DER=5.3%
[54/216] gofnj | DER=5.2%

Batch 1 done. 54 new files processed.
Total saved to CSV: 54


/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


In [ ]:
import csv
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/courtroom-diarizer-eval/results")
all_results = []

for i in range(1, 5):
    path = OUT_DIR / f"voxconverse_batch{i}.csv"
    if path.exists():
        with open(path) as f:
            rows = list(csv.DictReader(f))
            all_results.extend(rows)
            print(f"Batch {i}: {len(rows)} files loaded")
    else:
        print(f"Batch {i}: NOT FOUND")

# Write merged CSV
merged_path = OUT_DIR / "voxconverse_all_results.csv"
fields = ["file","duration_s","DER","miss","false_alarm","confusion"]
with open(merged_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(all_results)

print(f"\nTotal files merged: {len(all_results)}/216")
print(f"Saved → {merged_path}")

Batch 1: 54 files loaded
Batch 2: 54 files loaded
Batch 3: 54 files loaded
Batch 4: 54 files loaded

Total files merged: 216/216
Saved → /content/drive/MyDrive/courtroom-diarizer-eval/results/voxconverse_all_results.csv


In [ ]:
import csv
import numpy as np
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/courtroom-diarizer-eval/results")

with open(OUT_DIR / "voxconverse_all_results.csv") as f:
    all_results = list(csv.DictReader(f))

ders  = [float(r["DER"])         for r in all_results]
miss  = [float(r["miss"])        for r in all_results]
fa    = [float(r["false_alarm"]) for r in all_results]
conf  = [float(r["confusion"])   for r in all_results]

print(f"\n{'='*52}")
print(f"  VOXCONVERSE EVALUATION — pyannote auto (GPU)")
print(f"{'='*52}")
print(f"  Files evaluated  : {len(all_results)}/216")
print(f"  Mean DER         : {np.mean(ders):.2f}%")
print(f"  Median DER       : {np.median(ders):.2f}%")
print(f"  Std DER          : {np.std(ders):.2f}%")
print(f"  Min / Max DER    : {np.min(ders):.2f}% / {np.max(ders):.2f}%")
print(f"  Files < 10% DER  : {np.mean([d<10 for d in ders])*100:.1f}%")
print(f"  Files < 20% DER  : {np.mean([d<20 for d in ders])*100:.1f}%")
print(f"  Files < 35% DER  : {np.mean([d<35 for d in ders])*100:.1f}%")
print(f"{'='*52}")
print(f"  Error breakdown (mean across all files):")
print(f"  Miss             : {np.mean(miss):.2f}%")
print(f"  False alarm      : {np.mean(fa):.2f}%")
print(f"  Confusion        : {np.mean(conf):.2f}%")

# Top 5 worst files
print(f"\n  5 hardest files:")
sorted_results = sorted(all_results, key=lambda x: float(x["DER"]), reverse=True)
for r in sorted_results[:5]:
    print(f"    {r['file']:<10} DER={float(r['DER']):.1f}%  dur={float(r['duration_s']):.0f}s")

# Top 5 best files
print(f"\n  5 easiest files:")
for r in sorted_results[-5:]:
    print(f"    {r['file']:<10} DER={float(r['DER']):.1f}%  dur={float(r['duration_s']):.0f}s")


  VOXCONVERSE EVALUATION — pyannote auto (GPU)
  Files evaluated  : 216/216
  Mean DER         : 5.29%
  Median DER       : 5.34%
  Std DER          : 1.31%
  Min / Max DER    : 0.24% / 8.19%
  Files < 10% DER  : 100.0%
  Files < 20% DER  : 100.0%
  Files < 35% DER  : 100.0%
  Error breakdown (mean across all files):
  Miss             : 1.81%
  False alarm      : 3.01%
  Confusion        : 3.49%

  5 hardest files:
    vmaiq      DER=8.2%  dur=740s
    vbjlx      DER=8.1%  dur=324s
    vmbga      DER=8.0%  dur=660s
    vysqj      DER=7.9%  dur=89s
    wnfoi      DER=7.8%  dur=238s

  5 easiest files:
    mgpok      DER=1.9%  dur=564s
    tguxv      DER=1.7%  dur=266s
    tiams      DER=1.6%  dur=151s
    syiwe      DER=1.1%  dur=69s
    abjxc      DER=0.2%  dur=68s


In [ ]:
import csv
import numpy as np
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/courtroom-diarizer-eval/results")

with open(OUT_DIR / "voxconverse_all_results.csv") as f:
    results = list(csv.DictReader(f))


RTTM_DIR = Path("/content/drive/MyDrive/courtroom-diarizer-eval/annotations/dev")

fragmentation = 0
merging       = 0
exact         = 0

speaker_diffs = []

for r in results:
    rttm_path = RTTM_DIR / f"{r['file']}.rttm"
    if not rttm_path.exists():
        continue

    # Count reference speakers
    ref_speakers = set()
    with open(rttm_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 9 and parts[0] == "SPEAKER":
                ref_speakers.add(parts[7])

    ref_n   = len(ref_speakers)
    diff    = ref_n
    speaker_diffs.append(ref_n)

print("Reference speaker count distribution:")
from collections import Counter
counts = Counter(speaker_diffs)
for n, c in sorted(counts.items()):
    print(f"  {n} speakers: {c} files")

print(f"\nMean ref speakers: {np.mean(speaker_diffs):.1f}")
print(f"Max ref speakers:  {max(speaker_diffs)}")

Reference speaker count distribution:
  1 speakers: 22 files
  2 speakers: 44 files
  3 speakers: 35 files
  4 speakers: 24 files
  5 speakers: 31 files
  6 speakers: 17 files
  7 speakers: 12 files
  8 speakers: 11 files
  9 speakers: 4 files
  10 speakers: 6 files
  11 speakers: 3 files
  12 speakers: 3 files
  15 speakers: 2 files
  17 speakers: 1 files
  20 speakers: 1 files

Mean ref speakers: 4.5
Max ref speakers:  20


In [ ]:
import csv
import numpy as np
from pathlib import Path
from collections import defaultdict

OUT_DIR  = Path("/content/drive/MyDrive/courtroom-diarizer-eval/results")
RTTM_DIR = Path("/content/drive/MyDrive/courtroom-diarizer-eval/annotations/dev")

with open(OUT_DIR / "voxconverse_all_results.csv") as f:
    results = {r["file"]: r for r in csv.DictReader(f)}

# Group confusion rates by reference speaker count
confusion_by_spk = defaultdict(list)
der_by_spk       = defaultdict(list)

for filename, r in results.items():
    rttm_path = RTTM_DIR / f"{filename}.rttm"
    if not rttm_path.exists():
        continue
    ref_speakers = set()
    with open(rttm_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 9 and parts[0] == "SPEAKER":
                ref_speakers.add(parts[7])
    n = len(ref_speakers)
    confusion_by_spk[n].append(float(r["confusion"]))
    der_by_spk[n].append(float(r["DER"]))

print(f"{'Speakers':<10} {'Files':<7} {'Mean DER':<10} {'Mean Conf':<12} {'Mean Miss':<10}")
print("-" * 52)
for n in sorted(confusion_by_spk.keys()):
    confs = confusion_by_spk[n]
    ders  = der_by_spk[n]
    print(f"{n:<10} {len(confs):<7} "
          f"{np.mean(ders):<10.2f} "
          f"{np.mean(confs):<12.2f}")

Speakers   Files   Mean DER   Mean Conf    Mean Miss 
----------------------------------------------------
1          22      5.29       0.29        
2          44      5.29       1.24        
3          35      5.02       3.39        
4          24      5.30       4.03        
5          31      5.14       4.15        
6          17      5.47       4.32        
7          12      5.25       2.10        
8          11      5.25       10.49       
9          4       5.42       10.53       
10         6       6.08       5.60        
11         3       5.38       3.51        
12         3       6.66       10.18       
15         2       4.60       2.67        
17         1       8.19       4.58        
20         1       5.85       7.92        
